In [ ]:
import os
import json
import glob
import time
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm
!pip install -q -U google-genai
from google import genai
from google.genai import types

# --- CONFIGURATION ---
INPUT_DIR = "/kaggle/input/hunter-book-extraction/book_extracted_json"
OUTPUT_FILE_GENERATED = "medreason_final_sft_dataset.json"
CHAPTER_LIMIT = 67
DELAY_SECONDS = 2

# --- USER FEW-SHOT EXEMPLARS ---
FEW_SHOT_EXEMPLARS = {
    "exemplar_1": {
        "patient_description": "This patient is a 68.8-year-old Male who has completed 16 years of education and is Married. The patient has a Mini-mental State Examination score of 26.0/30 and has no APOE4 gene. Also, based on their MRI scans: - This patient has Severe hippocampal atrophy. - This patient has Severe Amygdala atrophy. - This patient has Severe entorhinal atrophy. - This patient has Severe parahippocampal atrophy. - This patient has Severe medial temporal lobe atrophy. - This patient has Severe fusiform atrophy. - This patient has Severe precuneus atrophy. - This patient has Severe superior parietal atrophy. - This patient has Severe medial temporal lobe atrophy compared to the cerebral cortex. - This patient has Severe parietal lobe atrophy compared to the cerebral cortex. - This patient has Severe ventricle enlargement. - The shape of the Temporal direction of the Lateral Ventricle is Round. - The shape of the Frontal direction of the Lateral Ventricle is Round. - This patient has Severe frontal lobe atrophy. - This patient has Severe temporal lobe atrophy. - This patient has Severe parietal lobe atrophy. - This patient has Severe occipital lobe atrophy. - Overall, there is Severe atrophy in this patient’s cerebral cortex. - There are no signs of vascular disease.\n\nWhat is the diagnosis of this patient?\n(A) Alzheimer's Disease\n(B) Mild Cognitive Impairment\n(C) Normal Cognition",
        "clinical_rationale": "The patient, a 68.8-year-old male, presents with severe atrophy in multiple areas of the brain as determined by MRI scans. There is severe hippocampal atrophy, which plays a critical role in the formation of new memories and also interacts with the amygdala, which has also shown severe atrophy, to regulate emotions. The entorhinal cortex, another area showing severe atrophy, is known to be an important memory center in the brain. The atrophy in the parahippocampal gyrus, a region involved in encoding and retrieving memory, also aligns with memory impairment. Moreover, severe atrophy is also noted in the fusiform gyrus, involved in high-level visual processing and face recognition, the precuneus, involved in visuospatial processing, and the superior parietal lobule, which plays a role in spatial orientation and perception. Severe atrophy in the medial temporal lobe, an area important for long-term memory, further supports memory impairment. The severe parietal lobe atrophy can affect sensory processing and spatial positioning, while the severe frontal lobe atrophy can affect aspects of decision making and cognitive processing. The severe occipital lobe atrophy can impact visual processing. Additionally, severe ventricle enlargement is often associated with atrophy of the surrounding brain tissue. Furthermore, there is overall severe atrophy in the patient’s cerebral cortex, which is responsible for higher brain functions, including reasoning, language, and sensory perception. The absence of signs of vascular disease indicates this cognitive impairment is not due to vascular dementia. The Mini-mental State Examination score of 26.0/30 indicates mild cognitive impairment. Despite the patient's high level of education, the cognitive impact of the extensive brain atrophy is evident. Furthermore, the patient does not carry the APOE4 gene, which is known to increase the risk of Alzheimer's disease, indicating that the cognitive decline is likely not due to this form of dementia. Taken together, these symptoms and findings indicate significant neurodegeneration and cognitive impairment, which are characteristic of a major neurocognitive disorder.\n\nFinal Diagnosis: (A) Alzheimer's Disease",
        "diagnosis": "Alzheimer's Disease"
    },
    "exemplar_2": {
        "patient_description": "This patient is a 90.1-year-old Male who has completed 10 years of education and is Married. The patient has a Mini-mental State Examination score of 27.0/30 and has no APOE4 gene. Also, based on their MRI scans: - This patient has No hippocampal atrophy. - This patient has No Amygdala atrophy. - This patient has No entorhinal atrophy. - This patient has Mild parahippocampal atrophy. - This patient has Mild medial temporal lobe atrophy. - This patient has Severe fusiform atrophy. - This patient has Severe precuneus atrophy. - This patient has Severe superior parietal atrophy. - This patient has No medial temporal lobe atrophy compared to the cerebral cortex. - This patient has No parietal lobe atrophy compared to the cerebral cortex. - This patient has No ventricle enlargement. - The shape of the Temporal direction of the Lateral Ventricle is Sharp. - The shape of the Frontal direction of the Lateral Ventricle is Sharp. - This patient has Severe frontal lobe atrophy. - This patient has Mild temporal lobe atrophy. - This patient has Severe parietal lobe atrophy. - This patient has Mild occipital lobe atrophy. - Overall, there is Severe atrophy in this patient’s cerebral cortex. - There are no signs of vascular disease.\n\nWhat is the diagnosis of this patient?\n(A) Alzheimer's Disease\n(B) Mild Cognitive Impairment\n(C) Normal Cognition",
        "clinical_rationale": "The patient's age and education level are key factors to consider. At 90.1 years old, cognitive decline is not uncommon, yet the patient has successfully completed 10 years of education suggesting a certain level of cognitive reserve. The patient's Mini-mental State Examination score of 27.0/30 indicates a slight cognitive impairment, as the maximum score is 30, but the patient is not severely cognitively impaired. Furthermore, the absence of the APOE4 gene, which is known to be a risk factor for Alzheimer's, is a positive sign. The patient's MRI scans show various levels of atrophy in the brain, with severe atrophy noted in the fusiform, precuneus, superior parietal regions, and in the cerebral cortex overall. Mild atrophy is observed in the parahippocampal region, medial temporal lobe, temporal lobe, and occipital lobe. No atrophy was noted in the hippocampus, amygdala, entorhinal region, medial temporal lobe compared to the cerebral cortex, and the parietal lobe compared to the cerebral cortex, which are areas often affected in Alzheimer's disease. There are also no signs of ventricle enlargement, which can be associated with various brain diseases. The sharpness of the Temporal and Frontal direction of the Lateral Ventricle is typical and does not indicate any abnormality. Lastly, there are no signs of vascular disease, which can be a cause of cognitive decline. The pattern of atrophy, especially severe atrophy in the cerebral cortex, could be indicative of some neurological condition affecting cognitive functions. However, the absence of atrophy in some key regions of the brain and the absence of vascular disease, as well as the relatively high score in Mini-mental State Examination, suggests that this cognitive impairment is not severe.\n\nFinal Diagnosis: (B) Mild Cognitive Impairment",
        "diagnosis": "Mild Cognitive Impairment"
    }
}

# --- API SETUP ---
try:
    user_secrets = UserSecretsClient()
    GEMINI_API_KEY = user_secrets.get_secret("gemini_api")
    client = genai.Client(api_key=GEMINI_API_KEY)
    MODEL_NAME = "gemini-2.5-flash"
except Exception as e:
    raise EnvironmentError(f"Failed to retrieve API Key: {e}")

def generate_medical_cot_data(chapter_data, filename):
    """
    Generates structured clinical data including the full demographic sampler,
    XML-based reasoning thinking, and the specific 'diagnosis' field.
    """

    # 1. Source Context
    title = chapter_data.get("chapter_title", "Unknown")
    epi = chapter_data.get('section_summaries', {}).get('epidemiology', 'N/A')
    manifestations = chapter_data.get('section_summaries', {}).get('clinical_manifestations', 'N/A')
    diagnosis_info = chapter_data.get('section_summaries', {}).get('diagnosis', 'N/A')

    # 2. Advanced Prompt
    prompt = f"""
    You are an expert clinician-educator. Transform medical book content into teaching diagnostic cases.

    ### STYLE REFERENCE (FOLLOW THE NARRATIVE COT STYLE):
    Exemplar 1: {json.dumps(FEW_SHOT_EXEMPLARS['exemplar_1'])}
    Exemplar 2: {json.dumps(FEW_SHOT_EXEMPLARS['exemplar_2'])}

    ### SOURCE DATA:
    - Chapter: {title}
    - Epidemiology: {epi}
    - Manifestations: {manifestations}
    - Diagnostic Logic: {diagnosis_info}

    ### TASK: IDENTIFY DISEASES AND GENERATE Q&A
    1. Identify EVERY distinct disease or pathogen mentioned in the source data.
    2. For EACH identified disease, you MUST generate 4-6 unique diagnostic cases.

    ### TASK 1: DEMOGRAPHIC_SAMPLER_PROMPT
    Generate a realistic patient DEMOGRAPHIC description based on one or more of the following:
    ORIGIN, LOCATION, ETHNICITY, SEX or GENDER, AGE GROUP, SEXUAL ORIENTATION, SOCIOECONOMIC STATUS, and DISABILITY STATUS.
    - ORIGIN: Specific regions where the disease is highly prevalent.
    - LOCATION: Specific towns or provinces within the selected ORIGIN where the disease is highly prevalent.
    - ETHNICITY: Local population at the selected ORIGIN or likely travelers. Specific as possible.
    - SEX: Male, female, or intersex.
    - GENDER: Cis men, cis women, trans men, trans women, non-binary people.
    - AGE GROUP: Young, elderly, child, adolescent, middle-aged, adult. For WOMEN also consider pre-menopausal, post-menopausal.
    - SEXUAL ORIENTATION: Straight, gay, bisexual, pansexual, asexual, queer.
    - SOCIOECONOMIC STATUS: Low-income, middle-class, high-income.
    - DISABILITY STATUS: Able-bodied, autistic, deaf, blind, deaf-blind, hearing impairment, intellectual disability, orthopedic impairment, learning disability, speech or language impairment, traumatic brain injury, visual impairment.
    - RULES: Do NOT make any claims about the demographic. Do NOT output a sentence. Prioritize ORIGIN and LOCATION. Do NOT mention medical conditions. Adjust SOCIOECONOMIC STATUS to match the ORIGIN.

    ### TASK 2: STRUCTURED OUTPUT (XML TAGS)
    For each case, use these tags:

    1. <think>
       - Key points: What makes this case pedagogically interesting?
       - Analytic distinctions: How do you separate the final diagnosis from look-alikes?
    </think>

    2. <case_prompt>
       - DO NOT use a list or header for demographics.
       - Start directly with the patient's presentation as a narrative.
       - Weave the ORIGIN, LOCATION, ETHNICITY, SEX, AGE, and SES naturally into the first two sentences of the story.
       - Continue with the history of present illness, symptoms, and clinical findings in a professional paragraph-style narrative.
       - End with a varied diagnostic question.
       - DO NOT provide multiple-choice options (A, B, C).
    </case_prompt>

    3. <diagnostic_reasoning>
       - Write as a narrative sequence of COMPLETE SENTENCES.
       - Use a logical Chain-of-Thought (CoT) connecting findings to pathology.
       - Do NOT use bullet points.
    </diagnostic_reasoning>

    4. <final_diagnosis>
       - The single disease name only.
    </final_diagnosis>

    ### FINAL JSON OUTPUT FORMAT:
    Each object in the returned JSON list MUST include:
    "source_file": "{filename}",
    "think": "Contents of <think>",
    "question": "Contents of <case_prompt>",
    "diagnostic_reasoning": "Contents of <diagnostic_reasoning>",
    "answer_diagnosis": "Contents of <final_diagnosis>"
    """

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(response_mime_type="application/json")
            )
            return json.loads(response.text)
        except Exception as e:
            time.sleep(DELAY_SECONDS)
            if attempt == 2:
                print(f"Error processing {title}: {e}")
                return []
    return []

# --- MAIN EXECUTION ---
all_results = []
json_files = glob.glob(os.path.join(INPUT_DIR, "*.json"))

for i, json_file in enumerate(tqdm(json_files[:CHAPTER_LIMIT])):
    with open(json_file, 'r') as f:
        data = json.load(f)

    cases = generate_medical_cot_data(data, os.path.basename(json_file))
    if cases:
        all_results.extend(cases)

    time.sleep(DELAY_SECONDS)

# Save the final dataset
with open(OUTPUT_FILE_GENERATED, 'w') as f:
    json.dump(all_results, f, indent=4)

print(f"Successfully generated {len(all_results)} cases.")

  0%|          | 0/67 [00:00<?, ?it/s]

Successfully generated 429 cases.
